In [ ]:
import numpy as np
from   scipy.optimize import root_scalar

# ---------- функция f(x; μ) = ∂Ω/∂x для y = 0 -------------------------------
def dOmega_dx(x, mu):
    """
    Первая производная эффективного потенциала CR3BP по x
    для коллинеарных точек (y = 0).
    """
    r1 = np.abs(x + mu)          # расстояние до m1
    r2 = np.abs(x - 1 + mu)      # расстояние до m2
    return x - (1 - mu)*(x + mu)/r1**3 - mu*(x - 1 + mu)/r2**3


# ---------- поиск L1, L2, L3 для одного значения μ --------------------------
def collinear_L_points(mu, tol=1e-12):
    f = lambda x: dOmega_dx(x, mu)

    # ---- L1 (между примариями, ближе к m2) ----------------------------------
    b = 1.0 - mu - 1e-12          # чуть левее второй примарии
    a = b - 0.8                   # начальная левая граница
    while f(a)*f(b) > 0:          # расширяем, пока не появится смена знака
        a -= 0.5
    L1 = root_scalar(f, bracket=(a, b), method='brentq',
                     xtol=tol, rtol=tol).root

    # ---- L2 (снаружи m2, x > 1-μ) -------------------------------------------
    a = 1.0 - mu + 1e-12          # чуть правее второй примарии
    b = a + 1.0
    while f(a)*f(b) > 0:
        b += 1.0
    L2 = root_scalar(f, bracket=(a, b), method='brentq',
                     xtol=tol, rtol=tol).root

    # ---- L3 (слева от m1, x < –μ) -------------------------------------------
    b = -mu - 1e-12               # чуть левее первой примарии
    a = b - 1.0
    while f(a)*f(b) > 0:
        a -= 1.0
    L3 = root_scalar(f, bracket=(a, b), method='brentq',
                     xtol=tol, rtol=tol).root

    return L1, L2, L3


# ---------- массивы значений -------------------------------------------------
N_uniform = 500
N_exp     = 100

mu_uniform = np.linspace(0.0, 0.5, N_uniform)
mu_exp     = np.geomspace(1e-12, mu_uniform[1], N_exp)   # ~[10⁻⁸, 1e-3]

mu_arr = np.unique(np.concatenate(([0.0], mu_exp, mu_uniform)))
M      = mu_arr.size                                    # общее число точек

L1_arr = np.empty(M)
L2_arr = np.empty(M)
L3_arr = np.empty(M)

for i, mu in enumerate(mu_arr):
    # μ = 0 даёт аналитические решения, чтобы не сталкиваться с 0/0
    if np.isclose(mu, 0.0, rtol=1e-14, atol=1e-14):
        L1_arr[i] =  1.0     # x = +1
        L2_arr[i] =  1.0
        L3_arr[i] = -1.0     # x = –1
        continue

    # остальные случаи – численное решение
    L1_arr[i], L2_arr[i], L3_arr[i] = collinear_L_points(mu)

# ---------- пример проверки --------------------------------------------------
mu_test = 0.012150584460351
idx     = np.argmin(np.abs(mu_arr - mu_test))
print('μ =', mu_arr[idx])
print('L1 =', L1_arr[idx])
print('L2 =', L2_arr[idx])
print('L3 =', L3_arr[idx])

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import math

In [ ]:
plt.plot(mu_arr, L1_arr, label=r'$L_1$', lw=2)
plt.plot(mu_arr, L2_arr, label=r'$L_2$', lw=2)
plt.plot(mu_arr, L3_arr, label=r'$L_3$', lw=2)
plt.grid()
plt.legend()
plt.ylabel('x, безразм.')
plt.xlabel(r'$\mu$')

# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L1-L3_point_x.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

In [ ]:
mu

In [ ]:
a = mu_arr

def L1_fun(x):
    y = (-0.66274804*x**0.32934248 - x**1.0944183 + math.cos(0.5164209*x**2.4841306)) #3.4e-05 #(0.0,0.5)
    return y
    
trues = L1_arr
preds = [L1_fun(x) for x in a]
preds = np.array(preds).flatten()

# Основной график
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(a, trues, label='Истинное значение', lw=2)
ax.plot(a, preds, '--', label='Аппроксимация', lw=2)
ax.grid()
plt.ylabel('x, безразм.')
plt.xlabel(r'$\mu$')
ax.legend(loc='lower left')

# Вложенный график (разница)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
axins = inset_axes(ax, width="40%", height="30%", loc='center right')
axins.plot(a, np.array(trues) - np.array(preds), color='black')
axins.set_title('Разница')
axins.grid()
axins.tick_params(labelsize=8)
axins.set_xlabel(r'$\mu$')
axins.set_ylabel('Δ, безразм.', fontsize=8)

# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L1_point_x.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:

plt.plot(mu_arr, L1_arr)
plt.plot(mu_arr, [L1_fun(x) for x in mu_arr],'--')
plt.grid()
plt.ylabel('x, безразм.')
plt.xlabel(r'$\mu$')
#plt.legend()
plt.show()   
print(L1_fun(0), L1_fun(0.5))

In [ ]:
def L2_fun(x):
    y = (0.73673844*0.8592316**x*(x**0.33944264 - x) + 1.0) #3.5e-05 #(0.0,0.5)
    return y
plt.plot(mu_arr, L2_arr)
plt.plot(mu_arr, [L2_fun(x) for x in mu_arr],'--')
plt.grid()
plt.ylabel('x, безразм.')
plt.xlabel(r'$\mu$')
#plt.legend()
plt.show()   
print()
print(L2_fun(0), L2_fun(0.5))

In [ ]:
from fractions import Fraction
print(Fraction(0.05409338).limit_denominator(100))

In [ ]:
2/37

In [ ]:
a = mu_arr

def L2_fun(x):
    y = (0.73673844*(0.8592316**x)*(x**0.33944264 - x) + 1.0) #3.5e-05 #(0.0,0.5)
    return y
    
trues = L2_arr
preds = [L2_fun(x) for x in a]
preds = np.array(preds).flatten()

# Основной график
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(a, trues, label='Истинное значение', lw=2)
ax.plot(a, preds, '--', label='Аппроксимация', lw=2)
ax.grid()
plt.ylabel('x, безразм.')
plt.xlabel(r'$\mu$')
ax.legend(loc='lower right')

# Вложенный график (разница)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
axins = inset_axes(ax, width="40%", height="30%", loc='center right')
axins.plot(a, np.array(trues) - np.array(preds), color='black')
axins.set_title('Разница')
axins.grid()
axins.tick_params(labelsize=8)
axins.set_xlabel(r'$\mu$')
axins.set_ylabel('Δ, безразм.', fontsize=8)

# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L2_point_x.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
def L3_fun(x):
    y = (0.05409338*x*math.exp(x*(0.646632818025174*x**2 + math.atan(x))) - 0.47076103*x - 1) #7.6e-08 #(0.0,0.5)
    return y
plt.plot(mu_arr, L3_arr)
plt.plot(mu_arr, [L3_fun(x) for x in mu_arr],'--')
plt.grid()
plt.ylabel('x, безразм.')
plt.xlabel(r'$\mu$')
#plt.legend()
plt.show()    
print(L3_fun(0), L3_fun(0.5))

In [ ]:
a = mu_arr

def L3_fun(x):
    #y = (0.05409338*x*math.exp(x*(0.646632818025174*x**2 + math.atan(x))) - 0.47076103*x - 1) #7.6e-08 #(0.0,0.5)
    y = (0.05408895*x*np.exp(x*(0.64677553*x**2 + np.arctan(x))) - 0.47075616*x - 1) #7.6e-08 #(0.0,0.5)
    return y
    
trues = L3_arr
preds = [L3_fun(x) for x in a]
preds = np.array(preds).flatten()

# Основной график
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(a, trues, label='Истинное значение', lw=2)
ax.plot(a, preds, '--', label='Аппроксимация', lw=2)
ax.grid()
plt.ylabel('x, безразм.')
plt.xlabel(r'$\mu$')
ax.legend(loc='lower left')

# Вложенный график (разница)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
axins = inset_axes(ax, width="40%", height="30%", loc='upper right')
axins.plot(a, np.array(trues) - np.array(preds), color='black')
axins.set_title('Разница')
axins.grid()
axins.tick_params(labelsize=8)
axins.set_xlabel(r'$\mu$')
axins.set_ylabel('Δ, безразм.', fontsize=8)

# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L3_point_x.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:

from scipy.optimize import least_squares

In [ ]:
L3_arr.shape

In [ ]:
mu_arr.shape

In [ ]:
def sklfun_base(x, mu):
    #y = (0.05409338*mu*np.exp(mu*(0.646632818025174*mu**2 + np.arctan(mu))) - 0.47076103*mu - 1) #7.6e-08 #(0.0,0.5)
    y = (x[0]*mu*np.exp(mu*(x[1]*mu**2 + np.arctan(mu))) - x[2]*mu - 1) #7.6e-08 #(0.0,0.5)
    y = (0.05408895*mu*np.exp(mu*(0.64677553*mu**2 + np.arctan(mu))) - 0.47075616*mu - 1) #7.6e-08 #(0.0,0.5)
    return y

def sklfun(x, mu, y):
    return sklfun_base(x, mu) - y

x0 = [(2/37),#0
      (11/17),#1
      (8/17),#2
     ]
true_array = L3_arr.flatten()
res_lsq = least_squares(sklfun, x0, args=(mu_arr.flatten(), L3_arr.flatten()))

pred_array = sklfun_base(res_lsq.x, mu_arr.flatten())
MSE = np.square(np.subtract(true_array,pred_array)).mean() 
RMSE_train = np.sqrt(MSE)
print('Train:',RMSE_train)
display(res_lsq.x)

In [ ]:
from fractions import Fraction
num = 0.47075616
print(Fraction(num).limit_denominator(1000))
print(Fraction(num).limit_denominator(1000)-num)

In [ ]:
plt.plot(mu_arr, true_array-pred_array)
plt.grid()

In [ ]:
mu_test = 0.012150584460351
print('L1 =', L1_fun(mu_test))
print('L2 =', L2_fun(mu_test))
print('L3 =', L3_fun(mu_test))

In [ ]:

L1 = 0.836915131427382 
L2 = 1.155682161024677 
L3 = -1.005062645331442

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.integrate import solve_ivp
import math
from   numpy.linalg    import norm

def odeCR3BP(t, p, mu):
    R = np.sqrt((p[0] + mu)**2 + p[1]**2)
    r = np.sqrt((p[0] - 1 + mu)**2 + p[1]**2)
    dpdt = np.array([p[2], p[3],
    2 * p[3] + p[0] - (1 - mu) * (p[0] + mu) / R**3 - mu * (p[0] - 1 + mu) / r**3,
    -2 * p[2] + p[1] - (1 - mu) * p[1] / R**3 - mu * p[1] / r**3])

    return dpdt
# ---------------------------------------------------------------------------
#  Проверка через odeCR3BP
# ---------------------------------------------------------------------------
def check_equilibrium(mu, x_point, tol=1e-3):
    """Возвращает True, если |(ax, ay)| <= tol."""
    _, _, ax, ay = odeCR3BP(0.0, (x_point, 0.0, 0.0, 0.0), mu)
    return norm([ax, ay]) <= tol

fails = []          # список (μ, название точки, модуль ускорения)


L1_arr_pred = [L1_fun(x) for x in mu_arr]
L2_arr_pred = [L2_fun(x) for x in mu_arr]
L3_arr_pred = [L3_fun(x) for x in mu_arr]
for mu, x1, x2, x3 in zip(mu_arr, L1_arr_pred, L2_arr_pred, L3_arr_pred):

    # Для μ = 0 проверка формально некорректна (m2 отсутствует) – пропускаем
    if np.isclose(mu, 0.0):
        continue

    for name, x in (('L1', x1), ('L2', x2), ('L3', x3)):
        _, _, ax, ay = odeCR3BP(0.0, (x, 0.0, 0.0, 0.0), mu)
        a_abs = norm([ax, ay])
        if a_abs > 1e-3:
            fails.append((mu, name, a_abs))

if not fails:
    print('Проверка пройдена: все точки дают |a| ≤ 1·10⁻¹¹')
else:
    print('Найдены отклонения:')
    for mu, name, a_abs in fails:
        print(f'  μ={mu:.3e}, {name}: |a| = {a_abs:.3e}')

In [ ]:
from ConvertFormula import expr_to_latex_with_consts

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(-0.66274804*x**0.32934248 - x**1.0944183 + math.cos(0.5164209*x**2.4841306)) #3.4e-05 #(0.0,0.5)"

litera = "A"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="L_1", j=1, litera=litera)
latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
expr_str = "(0.73673844*0.8592316**x*(x**0.33944264 - x) + 1.0) #3.5e-05 #(0.0,0.5)"

litera = "A"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="L_2", j=2, litera=litera)
latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
expr_str = "(0.05405405*x*np.exp(x*(0.64705882*x**2 + np.arctan(x))) - 0.47058824*x - 1) #7.6e-08 #(0.0,0.5)"

litera = "A"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="L_3", j=3, litera=litera)
latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
rows = [(i, v) for i, v in consts]  # ваши данные
tsv = '\n'.join(f'{v}' for i, v in rows)
print(tsv)